# Correlation and Fringe Fitting from the Ground Up [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/2026_eht-workshop/blob/main/tutorial.ipynb)

## Correlation

The correlation step takes signals from a pair of telescopes and
computes the Fourier transform of their cross correlations.
A significant property of such an operation is to remove the noise
from the visibility data.

Let $s$ be the sky signal from some radio source.
When the sky signal arrives at telescopes 1 and 2, the recorded
signals $s_1$ and $s_2$, although may have a time delay due to
geometric effects, are correlated.
Let $n_1$ and $n_2$ be noise at each of the telescopes, the cross
correlation is
\begin{align}
  X &= \langle (s_1 + n_1) (s_2 + n_2) \rangle \\
    &= \langle s_1 s_2 \rangle
     + \langle s_1 n_2 \rangle
     + \langle s_2 n_1 \rangle
     + \langle n_1 n_2 \rangle.
\end{align}
All the terms other than first term should vanishes.

By taking the Fourier transform of such a cross correlation, we obtain
also the spectral information of the signal.

In this chapter, we will create synthetic very long baseline
interferometry (VLBI) data and demostrate how cross-correlation can
remove the noise.
We will also show the XF and FX correlators are mathematically
identical, although the FX correlator is computationally more
efficient.

To get started, we first import the standard python packages:

In [ ]:
import numpy as np
from math import pi, ceil
from matplotlib import pyplot as plt

### Signal Generation

We consider monochromatic radio wave at unit frequency.

Using `numpy`, we create a time array `t` and then generate the
recorded signals `s1` and `s2` at the two telescopes.

The signal at telescope 2 has a lag of $1.2345/2\pi$ unit time
compared to telescope 1.

In [ ]:
t  = np.linspace(0, 10_000, num=100_000, endpoint=False)
s1 = np.sin(2 * pi * t)
s2 = np.sin(2 * pi * t - 1.2345)

Plotting the two signals,

In [ ]:
plt.plot(t[:20], s1[:20], label=r'$s_1$')
plt.plot(t[:20], s2[:20], label=r'$s_2$')
plt.legend()

### XF Correlator

Cross correlation is defined by:
\begin{align}
  X(f, g)(\tau) = \int f^*(t) g(t + \tau) dt = \int f^*(t - \tau) g(t) dt,
\end{align}
where $^*$ indicates complex conjugate.

For VLBI, we only care about the discrete version of this.
Hence we can replace $g(t + \tau)$ by `np.roll()`:

In [ ]:
tau = 1

print(np.roll(np.arange(10), tau)) # <- this is convolution
print(np.roll(np.arange(10),-tau)) # <- this is correlation

The cross correlation is simply:

In [ ]:
X = np.array([np.mean(s1 * np.roll(s2,-tau)) for tau in range(0,100_000)])

plt.plot(X[:20])

Applying the Fourier transform, we obtain the visibility as a function
of freqnecy:

In [ ]:
XF = np.fft.rfft(X)

plt.semilogy(abs(XF))

Pulling out the peak, the phase in the visibility is identical to the
lag we put in:

In [ ]:
n = np.argmax(abs(XF))
V = XF[n]

print(n)
print(abs(V))
print(np.angle(V))

### FX Correlator

Using the convolution theory, it is easy to show
\begin{align}
  \widehat{X(f, g)}_k = \hat{f}_k^* \hat{g}_k.
\end{align}
Hence, instead of first computing the cross correlation in time domain
and then applying the Fourier transform, we can perform the Fourier
transform first, and then compute the *element-wise* products in
frequency domain.
Correlators that use this methods are referred to as FX correlators,
which can we easily implement in python:

In [ ]:
S1 = np.fft.rfft(s1)
S2 = np.fft.rfft(s2)
FX = np.conj(S1) * S2

plt.semilogy(abs(FX))

Pulling out the peak, the phase in the visibility is identical to the
lag we put in, just like in the XF correlator:

In [ ]:
n = np.argmax(abs(FX))
V = FX[n]

print(n)
print(abs(V))
print(np.angle(V))

### Introducing Noise

We argued at the beginning that cross correlation removes noise from
the data.
To demostrate this, let's introduce noise in our simple python codes
with a signal-to-noise ratio at unity.

In [ ]:
n1 = np.random.normal(scale=np.sqrt(0.5), size=100_000)
n2 = np.random.normal(scale=np.sqrt(0.5), size=100_000)

print('SNR =', np.sum(s1*s1)/np.sum(n1*n1))
print('SNR =', np.sum(s2*s2)/np.sum(n2*n2))

plt.plot(s1 + n1)
plt.plot(s2 + n2)

Computing the visibility (spectrum) using the FX Correlator, we
immediate see the noise floor is almost 4 orders of magnitude lower
than the signal.

In [ ]:
S1 = np.fft.rfft(s1 + n1)
S2 = np.fft.rfft(s2 + n2)
FX = np.conj(S1) * S2

plt.semilogy(abs(FX))

And the error in the phase is at percentage level.

In [ ]:
n = np.argmax(abs(FX))
V = FX[n]

print(n)
print(abs(V))
print(np.angle(V))

### Chunked Correlation

In practice, correlators do not correlate all the data in a scan all
at once.
In stead, they divide the signal into multiple chunks, correlate each
chunk, and create a time series of visibility.
Such a time series can then be averaged later to incrase the
signal-to-noise ratio.

To demonstrate such process, let's choose a chunk size of 1000,
perform FX correlation over each chunk, and plot all of the resulting
visibilities as function of frequency.

In [ ]:
nc = 1000
Nc = int(ceil(len(s1) / nc) * nc)
S1 = np.fft.rfft(np.pad(s1+n1, (0, Nc-len(s1))).reshape(Nc//nc, nc))
S2 = np.fft.rfft(np.pad(s2+n2, (0, Nc-len(s2))).reshape(Nc//nc, nc))
FX = np.conj(S1) * S2

for fx in FX:
    plt.semilogy(abs(fx))

We may also look at only the "reference frequency" and plot the
amplitude and phase as function of time:

In [ ]:
n = np.argmax(abs(FX[0]))
V = FX[:,n]

print(n)

fig, (ax0, ax1) = plt.subplots(2,1, sharex=True)
fig.tight_layout()

ax0.plot(abs(V))
ax0.set_ylim(0, 300_000)

ax1.plot(np.angle(V))
ax1.set_ylim(-pi, pi)

We can also perform both the coherence and incoherence averaging of
the data:

In [ ]:
plt.semilogy(abs(np.mean(FX, axis=0)), label='coherence averaging')
plt.semilogy(np.mean(abs(FX), axis=0), label='incoherence averaging')
plt.legend()

## Calibration

The visibility data obtained by correlation contains systematic
errors.
Many of them shows up as delays, which translate to phase errors in
visibility.
Source of delays include:

* Water vapor in troposphere
* Electron content in Ionosphere
* Instruments
* Clock inaccuracy
* ..
* Thermal noise

Fringe-fitting is a major calibration step to remove the delay and
delay rates in the visibility data so they can be coherencely averaged
to higher signal-to-noise data.

To get a sense on how this works, let's set up our numerical
experiment.
We again need to import the standard python packages.
We also turn fix the noise realizations and put our correlator in a
function.

In [ ]:
import numpy as np
from math import pi, ceil
from matplotlib import pyplot as plt

t  = np.linspace(0, 10_000, num=100_000, endpoint=False)
n1 = np.random.normal(scale=1, size=100_000)
n2 = np.random.normal(scale=1, size=100_000)

def chunked_FX(s1, s2, n=1000):
    N  = int(ceil(len(s1) / n))
    S1 = np.fft.rfft(np.pad(s1+n1, (0, N*n-len(s1))).reshape(N, n))
    S2 = np.fft.rfft(np.pad(s2+n2, (0, N*n-len(s2))).reshape(N, n))
    return np.conj(S1) * S2

### Adding Delay and Delay Rate

To visualize how the delay and delay rate affect the visibility, we
define $d(t)$ and $r(t)$ and apply them to the signal for the second
station.

In [ ]:
d  = lambda t: 1e+3 * t/len(t) + 1
r  = lambda t: 1e-3 * t/len(t) + 1

s1 = np.sin(2 * pi * t)
s2 = np.sin(2 * pi * r(t) * (t + 0.123 / (2 * pi) * d(t)))

fig, (ax0, ax1, ax2) = plt.subplots(1,3, sharey=True, figsize=(12,4))
plt.subplots_adjust(wspace=0)
ax0.plot(t[:20], s1[:20])
ax0.plot(t[:20], s2[:20])
ax1.plot(t[50_000-10:50_000+10], s1[50_000-10:50_000+10])
ax1.plot(t[50_000-10:50_000+10], s2[50_000-10:50_000+10])
ax2.plot(t[-20:], s1[-20:], label=r'$s_1$')
ax2.plot(t[-20:], s2[-20:], label=r'$s_1$')
ax2.legend()

### Correlating the Signals

Using `chunked_FX()`, we can compute the visibility.

It has a "spectral" line at `freqnecy == 100` because the signal is
still monochromatic.

In [ ]:
FX = chunked_FX(s1, s2)

plt.imshow(abs(FX), origin='lower')
plt.xlabel('frequency')
plt.ylabel('time')

Focusing in our band at $\nu = 100$, it is easy to see the phase drift
away from a stable value.

In [ ]:
vis = FX[:,100]

fig, (ax0, ax1) = plt.subplots(2,1, sharex=True)
plt.subplots_adjust(hspace=0)

ax0.plot(abs(vis))
ax0.set_ylabel('Amplitude')
ax0.set_ylim(0, 300_000)

ax1.plot(np.angle(vis))
ax1.set_ylabel('Phase')
ax1.set_ylim(-pi, pi)
ax1.set_xlabel('time')

This phase drift affects coherence average.
When we average over long time scale, the coherencely averaged
amplitude drops, as shown in the following figure.
This is the reason incoherence averages are used sometimes.

In [ ]:
coh_avgs   = [abs(np.mean(vis[:i])) for i in range(1, 100+1)]
incoh_avgs = [np.mean(abs(vis[:i])) for i in range(1, 100+1)]

plt.plot(coh_avgs,   label='Coherence averages')
plt.plot(incoh_avgs, label='Incoherence averages')
plt.ylim(0, 300_000)
plt.xlabel('Averaging time')
plt.legend()

### Fringe-Fitting

The idea behind fringe-fitting is that we can expand the phase of an
arriving signal in terms of frequency and phase
\begin{align}
  \Delta\phi(\nu,t) \approx \phi_0 + \frac{\partial\phi}{\partial\nu}\Delta\nu + \frac{\partial\phi}{\partial t}\Delta t,
\end{align}
where $\phi_0$ is the phase error,
$\partial\phi/\partial\nu$ is the delay, and
$\partial\phi/\partial t$ is the (delay) rate.
Any process that estimate the delay and rate is considered
fringe-fitting.

For phase in the visibility, the idea is the same,
\begin{align}
  \Delta\phi_{12}(\nu, t)
  = \phi_{1,0} - \phi_{2,0}
  + (\frac{\partial\phi_1}{\partial\nu} - \frac{\partial\phi_2}{\partial\nu})\Delta\nu
  + (\frac{\partial\phi_1}{\partial t } - \frac{\partial\phi_2}{\partial t })\Delta t,
\end{align}

HOPS performs fringe fitting for each baseline independently, so
effectively it is fitting for the terms
$(\partial\phi_1/\partial\nu - \partial\phi_2/\partial\nu)$ and
$(\partial\phi_1/\partial t  - \partial\phi_2/\partial t )$.

In principle, we may use $\chi^2$-fitting to fit a straight line to
the phase, and de-rotate the visibility using such a fit.
In practice, however, because phase is a circular (or directional)
quantity, it is non-trivial to write a $\chi^2$-fitting method that
can work for arbitrary number of phase wraps.

Fortunately, Fourier transform has an interesting property that is
useful for fringe fitting.
Consider a function $f(t)$ and its Fourier transform $\hat{F}_\nu$.
When shifted by $t_0$ in time, the phase of the Fourier coefficient is
subjected to a linear rotation.
This can be easily proved.
Let $g(t) = f(t - t_0)$ and $\hat{G}_\nu$ be its Fourier transform,
\begin{align}
  \hat{G}_\nu
  &= \int g(t) e^{-2\pi\nu t} dt \\
  &= \int f(t - t_0) e^{-2\pi\nu t} dt \\
  &= \int f(t') e^{-2\pi\nu (t' + t_0)} dt'
   = \hat{F}_\nu e^{-2\pi\nu t_0}.
\end{align}
Therefore, it is possible to identify the linear part of phase drift
by solving for a shift in the Fourier transform of the visibility.

Here, we will remove the rate in our synthetic data.
We first perform Fourier transform of our visibility.
The `fftshift()` is not essential---it simply help plotting.

In [ ]:
Vis = np.fft.fftshift(np.fft.fft(vis))
R   = np.fft.fftshift(np.fft.fftfreq(100))
Rp  = R[np.argmax(abs(Vis))] # location of the peak in `Vis`

print(Rp)

plt.plot(R, abs(Vis))
plt.xlabel('spectral domain')
plt.ylabel('amplitude')
plt.axvline(Rp, color='r', linestyle=':')

With `Rp`, we can derotate the phase in the visiblity:

In [ ]:
viscal = vis * np.exp(-2j * pi * Rp * np.arange(len(vis)))

plt.plot(np.angle(vis),    label='correlated')
plt.plot(np.angle(viscal), label='calibrated')
plt.xlabel('time')
plt.ylabel('phase')
plt.legend()

Without the linear trend, we can perform coherence average for much longer time.

In [ ]:
vis_avgs    = [abs(np.mean(vis[:i]))    for i in range(1, 100+1)]
viscal_avgs = [abs(np.mean(viscal[:i])) for i in range(1, 100+1)]

plt.plot(vis_avgs,    label='Averaged correlated data')
plt.plot(viscal_avgs, label='Averaged calibrated data')
plt.ylim(0, 300_000)
plt.xlabel('Averaging time')
plt.legend()